# CodeGen — Group 45
## Step 1: Our Rust Execution Harness

**What we are building:** the "judge" for our project — a tool that takes a piece of Rust
code, **compiles and runs it against tests**, and reports pass / fail. We build this before
any AI, because every later experiment we run is scored by it.

**To reproduce our results:** `Runtime → Run all`. Sections 1–6 need **no GPU**.
Only the optional Section 7 (our real model baseline) needs a GPU
(`Runtime → Change runtime type → T4 GPU`).

**By the end we have:** a working harness, proof that it works, and our project's first real
number — the vanilla model's Rust score. That is essentially our Checkpoint 1.


## 1. Install the Rust toolchain
This gives us `rustc` (the Rust compiler). Takes ~1 minute.

In [ ]:
# Install Rust (non-interactive)
!curl https://sh.rustup.rs -sSf | sh -s -- -y -q

# Make rustc/cargo visible to this notebook
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

# Verify
!rustc --version
!cargo --version


warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.

  stable-x86_64-unknown-linux-gnu installed - rustc 1.96.0 (ac68faa20 2026-05-25)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh
rustc 1.96.0 (ac68fa

## 2. Install Python dependencies
Just the Hugging Face `datasets` library to download the benchmark.

In [ ]:
!pip install -q -U datasets huggingface_hub
print("done")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 125.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.1 MB/s eta 0:00:00
done


## 3. Load the MultiPL-E Rust problems
`humaneval-rs` = 156 classic coding problems, translated into Rust, **with unit tests**.
Each problem has:
- **prompt** — the function signature + a doc comment (ends with an open `{`)
- **tests** — a `fn main()` full of `assert_eq!` checks (starts with the closing `}`)

So a complete program is simply: **prompt + the model's body + tests**.

In [ ]:
from datasets import load_dataset

try:
    ds = load_dataset("nuprl/MultiPL-E", "humaneval-rs", split="test")
except Exception:
    ds = load_dataset("nuprl/MultiPL-E", "humaneval-rs", split="test", trust_remote_code=True)

print("Number of problems:", len(ds))
print("Fields:", ds.column_names)

# Look at one problem so the format is concrete
ex = ds[0]
print("\n===== PROMPT (given) =====\n", ex["prompt"])
print("===== TESTS (given) =====\n", ex["tests"])
print("===== stop tokens =====", ex["stop_tokens"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/33.2k [00:00<?, ?B/s]

humaneval-rs/test-00000-of-00001.parquet:   0%|          | 0.00/75.3k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/156 [00:00<?, ? examples/s]

Number of problems: 156
Fields: ['name', 'language', 'prompt', 'doctests', 'original', 'prompt_terminology', 'tests', 'stop_tokens']

===== PROMPT (given) =====
 /// Check if in given vector of numbers, are any two numbers closer to each other than
/// given threshold.
/// >>> has_close_elements(vec![1.0, 2.0, 3.0], 0.5)
/// false
/// >>> has_close_elements(vec![1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
/// true
fn has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool {

===== TESTS (given) =====
 }

fn main() {
    let candidate = has_close_elements;
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3), true);
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05), false);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.95), true);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.8), false);
    assert_eq!(candidate(vec![1.0, 2.0, 3.0, 4.0, 5.0, 2.0], 0.1), true);
    assert_eq!(candidate(vec![1.1, 2.2, 3.1, 4.1, 5.1], 1.0), true);
    ass

## 4. The harness function
This is the heart of Step 1. It glues the three parts into one `main.rs`, compiles it,
runs it, and returns one of: `pass`, `compile_error`, `run_fail`, `compile_timeout`, `run_timeout`.

In [ ]:
import subprocess, tempfile, os

def evaluate_one(prompt, completion, tests, compile_timeout=60, run_timeout=10):
    """Assemble prompt+completion+tests into a Rust program, compile and run it."""
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src  = os.path.join(wd, "main.rs")
        binp = os.path.join(wd, "prog")
        with open(src, "w") as f:
            f.write(program)

        # 1) compile
        try:
            c = subprocess.run(["rustc", src, "-o", binp],
                               capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "compile_timeout"
        if c.returncode != 0:
            return "compile_error"          # didn't even build

        # 2) run against the tests
        try:
            r = subprocess.run([binp], capture_output=True, text=True, timeout=run_timeout)
        except subprocess.TimeoutExpired:
            return "run_timeout"             # probably an infinite loop
        return "pass" if r.returncode == 0 else "run_fail"

print("harness ready")


harness ready


## 5. We self-test the harness (most important step)
Before we trust the harness, we prove it gives the right verdict on code we already know is
correct / wrong / broken. If these three checks don't come out as we expect, the bug is in our
**harness**, not in any model.

In [ ]:
ex = ds[0]   # HumanEval_0: has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool

# (a) a CORRECT body  -> should PASS
correct_body = """
    for i in 0..numbers.len() {
        for j in 0..numbers.len() {
            if i != j && (numbers[i] - numbers[j]).abs() < threshold {
                return true;
            }
        }
    }
    return false;
"""

# (b) a WRONG body (compiles, but fails the tests) -> should RUN_FAIL
wrong_body = "\n    return false;\n"

# (c) a BROKEN body (does not compile) -> should COMPILE_ERROR
broken_body = "\n    return this_is_not_defined;\n"

print("correct ->", evaluate_one(ex["prompt"], correct_body, ex["tests"]))
print("wrong   ->", evaluate_one(ex["prompt"], wrong_body,   ex["tests"]))
print("broken  ->", evaluate_one(ex["prompt"], broken_body,  ex["tests"]))

assert evaluate_one(ex["prompt"], correct_body, ex["tests"]) == "pass"
assert evaluate_one(ex["prompt"], wrong_body,   ex["tests"]) == "run_fail"
assert evaluate_one(ex["prompt"], broken_body,  ex["tests"]) == "compile_error"
print("\nHarness works correctly — it can tell good Rust from bad.")


correct -> pass
wrong   -> run_fail
broken  -> compile_error

Harness works correctly — it can tell good Rust from bad.


## 6. We run the harness over ALL problems (end-to-end pipeline test)
Here we feed a **dummy** body (`todo!()`) to every problem. It compiles but panics at runtime,
so almost everything comes back `run_fail`. The point isn't the score — it's that we prove our
harness runs cleanly across all 156 problems and gives us aggregate counts.

In [ ]:
from collections import Counter

def run_benchmark(completion_fn, limit=None):
    """completion_fn(example) -> a Rust function body (string)."""
    statuses = []
    data = ds if limit is None else ds.select(range(limit))
    for ex in data:
        body = completion_fn(ex)
        statuses.append(evaluate_one(ex["prompt"], body, ex["tests"]))
    counts = Counter(statuses)
    pass_rate = counts["pass"] / len(statuses)
    return pass_rate, counts

# Dummy "model": always returns todo!()  (compiles, panics at runtime)
dummy_rate, dummy_counts = run_benchmark(lambda ex: "\n    todo!()\n")
print("Dummy completion — execution accuracy:", round(100*dummy_rate, 1), "%")
print("Breakdown:", dict(dummy_counts))


Dummy completion — execution accuracy: 0.0 %
Breakdown: {'run_fail': 156}


## 7. Our REAL baseline — the vanilla model
Here we load `codegen-350M-multi` and let it actually attempt the Rust problems, giving us the
**baseline number we improve on later**. We expect it to be low — that's the whole point.

Needs a **GPU runtime** (`Runtime → Change runtime type → T4 GPU`). We run the first 20
problems for speed; we can remove `limit=20` to score all 156.

In [ ]:
!pip install -q transformers accelerate torch

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Salesforce/codegen-350M-multi"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
print("model loaded on", model.device)

def model_completion(ex, max_new_tokens=256):
    prompt = ex["prompt"]
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    # cut at the earliest stop token (e.g. "\n}") so we keep only the function body
    cut = len(text)
    for s in ex["stop_tokens"]:
        i = text.find(s)
        if i != -1:
            cut = min(cut, i)
    return text[:cut]

rate, counts = run_benchmark(model_completion)
print("\nVanilla codegen-350M-multi on Rust:", round(100*rate, 1), "%")
print("Breakdown:", dict(counts))
print("\nThis low number is our BASELINE. Fine-tuning + RAG aim to push it up.")


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/797M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/797M [00:00<?, ?B/s]

model loaded on cuda:0

Vanilla codegen-350M-multi on Rust: 1.3 %
Breakdown: {'compile_error': 144, 'run_fail': 10, 'pass': 2}

This low number is our BASELINE. Fine-tuning + RAG aim to push it up.


In [ ]:
# Quick sanity check: look at ONE real example end-to-end
ex = ds[3]
body = model_completion(ex)
program = ex["prompt"] + body + ex["tests"]

print("=== WHAT THE MODEL WROTE (the body) ===")
print(body)
print("\n=== FULL ASSEMBLED PROGRAM ===")
print(program)

# Show the actual compiler error
import subprocess, tempfile, os
with tempfile.TemporaryDirectory() as wd:
    src = os.path.join(wd, "main.rs"); open(src, "w").write(program)
    c = subprocess.run(["rustc", src, "-o", os.path.join(wd, "prog")],
                       capture_output=True, text=True)
    print("\n=== RUSTC ERROR ===")
    print(c.stderr[:1500])

=== WHAT THE MODEL WROTE (the body) ===
    return operations.length() == 0;

=== FULL ASSEMBLED PROGRAM ===
/// You're given a vector of deposit and withdrawal operations on a bank account that starts with
/// zero balance. Your task is to detect if at any point the balance of account fallls below zero, and
/// at that point function should return true. Otherwise it should return false.
/// >>> below_zero(vec![1, 2, 3])
/// false
/// >>> below_zero(vec![1, 2, -4, 5])
/// true
fn below_zero(operations: Vec<isize>) -> bool {
    return operations.length() == 0;}

fn main() {
    let candidate = below_zero;
    assert_eq!(candidate(Vec::<isize>::new()), false);
    assert_eq!(candidate(vec![1, 2, -3, 1, 2, -3]), false);
    assert_eq!(candidate(vec![1, 2, -4, 5, 6]), true);
    assert_eq!(candidate(vec![1, -1, 2, -2, 5, -5, 4, -4]), false);
    assert_eq!(candidate(vec![1, -1, 2, -2, 5, -5, 4, -5]), true);
    assert_eq!(candidate(vec![1, -2, 2, -2, 5, -5, 4, -4]), true);
}


=== RUSTC E

## What we built
- A **Rust execution harness** that scores code by compiling and running it.
- **Proof** it works (Section 5) and that it runs over the whole benchmark (Section 6).
- Our project's **first real metric**: the vanilla model's Rust score (Section 7).

**This is our Checkpoint 1.** From here we curate Python→Rust training data, then fine-tune
with LoRA and re-run `run_benchmark(...)` to measure our improvement.

**Our convention:** we keep `evaluate_one` and `run_benchmark` unchanged — every future
experiment (our fine-tuned model, +RAG, the big LLM) just plugs a different `completion_fn`
into `run_benchmark`, so all our numbers stay comparable.